## 🧠 TinyStories Language Model – A 15M Parameter GPT from Scratch

This project is an educational implementation of a **small-scale GPT-style language model**, built entirely from scratch using **PyTorch** and trained on the **TinyStories** dataset introduced by Apple.

The goal is to understand the internal workings of large language models (LLMs) by developing a **15 million parameter Transformer**, taking architectural inspiration from **GPT-3**. The TinyStories dataset is ideal for this purpose due to its simplicity, small size, and well-formed language structure.

Target:
A model that make stories for children and those stories should produce coherent text, stories should convey some meaning and the model should be able to understand correct grammer for some far.


### 📚 References
- 📄 [TinyStories: How Small Can Language Models Be and Still Speak Coherently? (Apple, 2023)](https://arxiv.org/abs/2305.07759)
- 📄 [Language Models are Few-Shot Learners (GPT-3 paper)](https://arxiv.org/abs/2005.14165)


### PART-1 : EXPLORING THE DATASET

📚 Dataset: TinyStories

The TinyStories dataset is a collection of short, synthetically generated children's stories created using GPT-3.5 and filtered for coherence, simplicity, and grammatical correctness. It provides a compact, high-quality corpus ideal for training small language models in a low-compute environment. Each story is typically 200–300 tokens long and written in simple, well-structured English.

In [13]:
# Tiny Stories dataset can be found in huggingface
# Those datasets can imported using dataset lib

from datasets import load_dataset

ds = load_dataset('roneneldan/TinyStories')

In [14]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})


In [15]:
print(ds['train'].features)

{'text': Value(dtype='string', id=None)}


In [16]:
for i in range(3):
    print(ds['train'][i])


{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}
{'text': 'Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.\n\nOne day, Beep was driving in the park when he saw a big tree. The tree had man

### PART-2 : DATA PREPROCESSING

## 🔤 Byte Pair Encoding (BPE) – Quick Reference

### 🔧 What is BPE?
Byte Pair Encoding (BPE) is a **subword tokenization** method that sits between word-level and character-level encoding. It merges the most frequent pairs of characters or subwords to form new tokens, allowing the model to:
- Keep vocabulary small,
- Handle unknown words effectively,
- Shorten input sequences compared to character-level tokenization.

---

### 🚧 Why Use BPE?
- ✅ Compact vocabulary → faster and more memory-efficient training
- ✅ Robust to rare and unseen words
- ✅ Shorter token sequences than character-level
- ✅ Proven in LLMs like GPT-2 and GPT-3

---

### ❌ Why Not Word-Level or Character-Level?

#### ❌ Word-Level Encoding
- 🧨 **Huge vocabulary** → requires more memory and larger embedding layers
- 📉 **Fails on rare/misspelled/compound words** (OOV problem)
- 🚫 **Not scalable** for LLM training

#### ❌ Character-Level Encoding
- 🧪 **Simpler to implement**, but...
- 🐢 **Leads to long token sequences** → slower training and inference
- ❌ **Harder for model to learn meaningful representations**

---

### ⚙️ How BPE Works
1. Start with character-level tokens (e.g., `s t o r y </w>`).
2. Count the frequency of all adjacent token pairs (e.g., `t h`, `h e`, etc.).
3. Merge the most frequent pair into a new subword token (e.g., `th`, `he`).
4. Repeat until desired vocabulary size is reached (e.g., 5,000–8,000 tokens).

---

### 🧠 Example
Input sentence:  
`the ship is sinking`

Initial tokens:  
`t h e</w>` `s h i p</w>` `i s</w>` `s i n k i n g</w>`

After frequent merges:  
**`the` `ship` `is` `sink` `ing`**

---

### 🛠️ Tooling
- 📦 Using tiktoken library (pip install tiktoken)

---

### 📌 Practical Setup for This Project
- 🧠 Train a custom BPE tokenizer on TinyStories
- 🧰 Target vocab size: **~5,000–8,000**
- 📏 Max sequence length: **128–256 tokens**

---


In [ ]:
import tiktoken
import os
import numpy as np
from tqdm.auto import tqdm

# Creates an encoder instance for the GPT-2 tokenizer.
enc = tiktoken.get_encoding("gpt2")

# Encodes the text into token IDs using encode_ordinary
# (which encodes text without special tokens).
def process(example):
    import tiktoken
    enc = tiktoken.get_encoding("gpt2")
    ids = enc.encode_ordinary(example['text'])  # encode_ordinary ignores special tokens
    out = {'ids': ids, 'len': len(ids)}
    return out


In [25]:
sample = {"text": "Once upon a time in a tiny land."}
print(process(sample))

{'ids': [7454, 2402, 257, 640, 287, 257, 7009, 1956, 13], 'len': 9}


In [28]:
if not os.path.exists("train.bin"):
    tokenized = ds.map(
        process,
        remove_columns=['text'],
        desc="tokenizing the splits",
        num_proc=2,
        )
    # concatenate all the ids in each dataset into one large file we can use for training
    for split, dset in tokenized.items():
        arr_len = np.sum(dset['len'], dtype=np.uint64)
        filename = f'{split}.bin'
        dtype = np.uint16 # (can do since enc.max_token_value == 50256 is < 2**16)
        arr = np.memmap(filename, dtype=dtype, mode='w+', shape=(arr_len,))
        total_batches = 1024

        idx = 0
        for batch_idx in tqdm(range(total_batches), desc=f'writing {filename}'):
            # Batch together samples for faster write
            batch = dset.shard(num_shards=total_batches, index=batch_idx, contiguous=True).with_format('numpy')
            arr_batch = np.concatenate(batch['ids'])
            # Write into mmap
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)
        arr.flush()

tokenizing the splits (num_proc=2):   0%|          | 0/2119719 [00:00<?, ? examples/s]

NameError: name 'enc' is not defined